In [1]:
import pandas as pd
import os
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

d:\Program\envs\agent_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
C:\Users\ANH TU\AppData\Local\Temp\ipykernel_22708\601060176.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
csv_file_path = r"D:\PharmaRAG-VN\Data\Clean\All_Documents_chunk.csv"
vector_db_path = r"D:\PharmaRAG-VN\Data\VectorStore\faiss_index"

In [3]:
df = pd.read_csv(csv_file_path)

In [4]:
df = df.fillna("")

In [5]:
documents = []

for _, row in df.iterrows():

    metadata = {
        "document_type": row["document_type"]
    }

    if row["level_1"]:
        metadata["level_1"] = row["level_1"]

    if row["level_2"]:
        metadata["level_2"] = row["level_2"]

    if row["level_3"]:
        metadata["level_3"] = row["level_3"]

    sections = []

    if row["document_type"]:
        sections.append(f"Document Type: {row['document_type']}")

    if row["level_1"]:
        sections.append(f"Level 1: {row['level_1']}")

    if row["level_2"]:
        sections.append(f"Level 2: {row['level_2']}")

    if row["level_3"]:
        sections.append(f"Level 3: {row['level_3']}")

    sections.append(f"Content:\n{row['content']}")

    embedding_text = "\n".join(sections)

    documents.append(
        Document(
            page_content=embedding_text,
            metadata=metadata
        )
    )

In [6]:
len(documents)

10032

In [7]:
documents = documents[:50]

In [8]:
len(documents)

50

In [9]:
documents[49].metadata

{'document_type': 'guideline',
 'level_1': '**HƯỚNG DẪN SỬ DỤNG DƯỢC THƯ QUỐC GIA VIỆT NAM**',
 'level_2': '**SỬ DỤNG HỢP LÝ THUỐC KHÁNG ĐỘNG KINH**',
 'level_3': '**Kết luận**'}

In [10]:
print(documents[49].page_content)

Document Type: guideline
Level 1: **HƯỚNG DẪN SỬ DỤNG DƯỢC THƯ QUỐC GIA VIỆT NAM**
Level 2: **SỬ DỤNG HỢP LÝ THUỐC KHÁNG ĐỘNG KINH**
Level 3: **Kết luận**
Content:
Tiêu chuẩn chủ yếu của hiệu quả điều trị là kiểm soát được tối ưu các cơn động kinh. Trên nguyên tắc, thầy thuốc điều trị phải theo dõi diễn biến lâm sàng trong quá trình người bệnh được dùng thuốc kháng động kinh. Ngoài việc kiểm tra lâm sàng và điện não đồ, cần theo dõi thường quy các xét nghiệm sinh học, đặc biệt chú ý tới các thông số gan, thận và các chỉ số huyết học. Ở một số trung tâm y tế chuyên khoa có trang thiết bị phù hợp, có thể tiến hành định lượng thuốc kháng động kinh trong huyết tương vài tuần sau khi bắt dầu dùng thuốc và sau đó mỗi năm từ 1 - 2 lần.


In [11]:
embeddings = OpenAIEmbeddings(
        model="text-embedding-3-large",
    )

In [12]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [13]:
os.makedirs(os.path.dirname(vector_db_path), exist_ok=True)
vectorstore.save_local(vector_db_path)